# Route labels — the golden set over the cell composition

**Input.** `CellFill().build()` → `data/composition/cell_selection.parquet`: the
query set the 44 archetype cells (`cells.yaml`) selected, ~51.5K distinct
queries across 42 datasets. That is the dataset; nothing here invents queries.

**Output.** `data/route_labels/labels.parquet`: **one row per labelled
`(dataset, query_id)`** carrying the *route label* — which of `dense_only`,
`pure_rrf`, `sparse_only` retrieved best — the losing routes' scores, the
outcome shape, and the selection's own `cell` / `checkable` columns. A query
that fills several cells is labelled once; per-cell readouts join back to
`cell_selection` on `query_id`.

**How the label is decided.** All three routes are run, each ranking scored by
the router objective, argmax wins. No LLM is asked which route is better:
dense-vs-sparse depends on the corpus vocabulary and its IDF, neither of which
is in the query, so a model reading only the query is being asked to predict
something that isn't a function of its input.

**Coverage.** Every one of the 42 datasets has its corpus materialized on disk
(this session's `materialize_corpora.py` run), so the only gap between selected
and labelled is Qdrant indexing — closed by the size-ascending sweep in §17.
The anchor sections (§4–§15) walk two datasets in detail first; the sweep then
labels the rest. `min_relevance` is per-lane from `LANES` (antique 3, trec-dl 2,
the rest 1), so graded lanes are binarized at the grade their benchmark intends.

**Prerequisites**

```bash
docker compose up -d          # local Qdrant on :6333
```

In [ ]:
%load_ext autoreload
%autoreload 2

## 1 — Setup

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from qdrant_client import QdrantClient
from qdrant_client.models import Distance

load_dotenv(".env") or load_dotenv("../.env")

DENSE_MODEL, DENSE_SIZE = "BAAI/bge-small-en-v1.5", 384
SPARSE_MODEL = "Qdrant/bm25"

client = QdrantClient(
    url=os.getenv("QDRANT_URL", "http://localhost:6333"),
    api_key=os.getenv("QDRANT_API_KEY"),
)


def show(df: pd.DataFrame) -> None:
    display(Markdown(df.to_markdown(index=False)))


print("qdrant collections:", [c.name for c in client.get_collections().collections])

## 2 — Load the composition

`CellFill().build()` returns the cell selection, reading
`cell_selection.parquet` from disk when it exists — so this does not re-run the
fill. `RouteLabels` is selection-agnostic: it carries whichever of
`cell` / `stage` / `route` / `checkable` the frame has.

In [ ]:
from composition import CellFill
from hybrid_search_rrf_dataset.labels import RouteLabels
from hybrid_search_rrf_dataset.objective import RouterObjective

selection = CellFill().build()   # data/composition/cell_selection.parquet — the 44-cell query set
print(f"selection: {len(selection):,} rows x {len(selection.columns)} cols "
      f"({selection[['dataset','query_id']].drop_duplicates().shape[0]:,} distinct queries)")
show(selection.sample(5))

labels = RouteLabels(selection, objective=RouterObjective(min_relevance=1))
print("objective:", labels.objective.name)

## 3 — What can be labelled today

`unlabelled` means the row is in the composition but its dataset has no
indexed corpus or no qrels on disk. This table is the real progress bar for
the golden set, and it should be re-read after every dataset lands.

In [ ]:
coverage = labels.coverage()
show(coverage)
print(f"selected {coverage.selected.sum():,} | "
      f"labelled {coverage.labelled.sum():,} | "
      f"unlabelled {coverage.unlabelled.sum():,} "
      f"({coverage.unlabelled.sum() / coverage.selected.sum() * 100:.1f}%)")

## 4 — Index one dataset's corpus

Starting with `beir-nfcorpus`: 3,633 documents, small enough to index in full
with no corpus sampling, and it ships human judgments so the labels are
checkable. It is also richly judged — 38.2 judged docs per query, and 300 of its
323 queries have two or more relevant documents, which is the only regime where
the choice of objective can change a label at all.

**Not** reusing the existing `nf` collection: it holds 33,633 points — 3,633
nfcorpus documents mixed with 30,000 trec-dl passages — so nfcorpus queries
would be scored against a corpus that is 90% unrelated.

Dense and sparse live as two named vector slots on one collection, which is what
lets a single Qdrant call fuse them. Idempotent: re-running skips the upload.

In [ ]:
from hybrid_search_rrf_dataset.indexer import (
    CorpusDocument,
    CorpusIndexer,
    EmbeddingCache,
    EmbeddingConfig,
)
from hybrid_search_rrf_dataset.retrieval import SnapshotDataset

DATASET = "beir-nfcorpus"        # the composition's key
COLLECTION = "nfcorpus_routes"

# snapshot dir is the lane's source.name (beir-nfcorpus), not the upstream slug
source = SnapshotDataset(DATASET, path="data")   # written by RetrievalDataset.save()
corpus = source.corpus()

dense_cfg = EmbeddingConfig(
    name="dense_base", model_id=DENSE_MODEL, kind="dense",
    size=DENSE_SIZE, distance=Distance.COSINE,
    parallel=4
)
sparse_cfg = EmbeddingConfig(name="sparse_base", model_id=SPARSE_MODEL, kind="sparse")

indexer = CorpusIndexer(
    client, COLLECTION,
    embeddings=[dense_cfg, sparse_cfg],
    cache=EmbeddingCache("./.embedding_cache"),
)
indexer.ensure_collection()

if client.count(COLLECTION, exact=True).count >= len(corpus):
    print(f"{COLLECTION}: already indexed — skipping upload")
else:
    indexer.upload([CorpusDocument(**r) for r in corpus.to_dict("records")], batch_size=64)
print(f"{COLLECTION}: {client.count(COLLECTION, exact=True).count:,} points")

## 5 — The three routes

`DenseOnlyStrategy` and `SparseOnlyStrategy` keep their raw scores;
`PureRRFStrategy` is Qdrant-native RRF, matching production's `Fusion::Rrf`.

RRF fuses by *rank position*, which is exactly why it can lose on top-1: a doc
ranked first by dense and 40th by sparse loses to one ranked third by both.

In [ ]:
from hybrid_search_rrf_dataset.fusion import (
    DenseOnlyStrategy,
    PureRRFStrategy,
    SparseOnlyStrategy,
)

args = (client, COLLECTION, dense_cfg, sparse_cfg)
dense, hybrid, sparse = (
    DenseOnlyStrategy(*args), PureRRFStrategy(*args), SparseOnlyStrategy(*args)
)

probe = labels.rows_for(DATASET)["query"].iloc[0]
print(f"probe (a real selection row): {probe!r}\n")
for s in (dense, hybrid, sparse):
    top = list(s.rank(probe).items())[:3]
    print(f"  {s.name:12s} " + "  ".join(f"{d}={v:.3f}" for d, v in top))

## 6 — Label this dataset's selection rows

`RouteLabels.label` narrows the source to the composition's query ids via
`QuerySubset`, runs `GoldenRoutingBuilder`, and merges the result into
`labels.parquet` — replacing only this dataset's rows.

The objective is `0.7·HitRate@1 + 0.3·NDCG@10`. Because the hit weight exceeds
the NDCG weight the score ranges are disjoint (rank-1 hit ⇒ ≥0.700, miss ⇒
≤0.300), so it is lexicographic: top-1 decides, NDCG@10 only breaks ties within
each group. `min_relevance=1` because nfcorpus grades are 1 and 2 only.

In [ ]:
labelled = labels.label(source, dense, hybrid, sparse, dataset=DATASET)
print(f"labelled {len(labelled):,} rows -> {labels.labels_path}")
show(labelled.head(8).round(3))

## 7 — Outcome shapes

Only one of the three shapes teaches the router about dense-vs-sparse.

| shape | meaning |
| --- | --- |
| `routes_differ` | the quality signal — the trainable set |
| `all_tied` | any route works; serve the cheapest. Signal for the *speed* goal |
| `all_zero` | nothing relevant found by any route — unanswerable, and **no valid label exists**. `route` is stored null |

Read per **cell** — the archetype the query was selected into. The carried
`cell` is one representative when a query fills several; the exact per-cell
view joins `labels` back to `cell_selection` on `query_id` so a multi-cell
query counts under each of its cells. The empty `cell` is the control /
feature-blind draw.

In [ ]:
shapes = (labelled.groupby(["cell", "shape"]).size()
          .unstack(fill_value=0))
shapes["total"] = shapes.sum(axis=1)
show(shapes.reset_index())

overall = labelled["shape"].value_counts().rename_axis("shape").reset_index(name="rows")
overall["share"] = (overall.rows / len(labelled) * 100).round(1).astype(str) + "%"
show(overall)

signal = labelled[labelled["shape"] == "routes_differ"]
dist = signal.route.value_counts().rename_axis("route").reset_index(name="rows")
dist["share"] = (dist.rows / len(signal) * 100).round(1).astype(str) + "%"
print(f"route distribution over the {len(signal):,} trainable rows:")
show(dist)
print(f"majority-class baseline: {dist.rows.iloc[0] / len(signal) * 100:.0f}% "
      f"(always predict {dist.route.iloc[0]})")

## 8 — Constant-route baselines

The bar a router must clear is **the best constant route**, not random. A
candidate can post respectable regret against the oracle and still lose to one
line of code, so these belong in every comparison.

In [ ]:
from hybrid_search_rrf_dataset.evaluation import compare
from hybrid_search_rrf_dataset.golden import BaselineBuilder, GoldenRoutingBuilder
from hybrid_search_rrf_dataset.retrieval import QuerySubset

subset = QuerySubset(source, labels.rows_for(DATASET)["query_id"])
oracle = GoldenRoutingBuilder(
    dense, hybrid, sparse, objective=labels.objective
).build_or_load(subset, path=f"data/route_labels/{DATASET}_oracle")

rows = []
for strategy in (dense, hybrid, sparse):
    const = BaselineBuilder(strategy, objective=labels.objective).build_or_load(
        subset, path=f"data/route_labels/{DATASET}_const_{strategy.name}"
    )
    s = compare(oracle, const)
    rows.append({
        "always pick": str(strategy.name),
        "mean score": round(s.mean_metric_candidate, 3),
        "oracle ceiling": round(s.mean_metric_golden, 3),
        "mean regret": round(s.mean_regret, 3),
        "p90 regret": round(s.p90_regret, 3),
        "reaches oracle": f"{s.oracle_hit_rate_pct:.0f}%",
        "route agreement": f"{s.route_agreement_pct:.0f}%",
    })
show(pd.DataFrame(rows))

## 9 — Objective sensitivity, at zero retrieval cost — pooled across every lane

`route_rankings` holds each route's top-10 doc ids and the judgments live in
`QrelStore`, so any metric depending only on the *order* of the top 10 can be
recomputed without touching Qdrant. That is what makes the objective a
reversible decision instead of a one-way door.

Originally this cell recomputed for the single nfcorpus anchor dataset only.
It now pools across **every lane with a cached oracle** —
`data/route_labels/{lane}_oracle/rows.parquet`, one retrieval pass per lane,
built once via `GoldenRoutingBuilder.build_or_load`. Each variant threads the
*lane's own* `min_relevance` through (antique binarizes at grade 3, trec-dl at
2, the rest at 1) rather than a single flat value — the "shipped" column must
reproduce exactly what `labels.parquet` actually stores, or the flip-rate
comparison is comparing against a fiction.

We synthesize descending scores from the stored order — HitRate@1 and NDCG@10
depend only on that order, so the recomputation is exact. It cannot evaluate a
metric needing rank 11+ or the raw retrieval scores.

Safe to run before every lane finishes caching: an uncached lane is skipped
and counted, not a crash — re-run later for fuller coverage.

In [ ]:
from pathlib import Path

from hybrid_search_rrf_dataset.fusion import derive_route
from hybrid_search_rrf_dataset.golden import GoldenRoutingBuilder
from hybrid_search_rrf_dataset.lanes import LANES
from hybrid_search_rrf_dataset.objective import NDCGObjective
from hybrid_search_rrf_dataset.qrels import QrelStore
from hybrid_search_rrf_dataset.retrieval import SnapshotDataset


def load_lane_oracle(key: str):
    path = Path("data/route_labels") / f"{key}_oracle"
    if not (path / "rows.parquet").exists():
        return None
    return GoldenRoutingBuilder.load(path=path)


lane_rows, lane_lookup, skipped = {}, {}, []
for key in LANES:
    rows = load_lane_oracle(key)
    if rows is None:
        skipped.append(key)
        print(f"{key:40s} skip — not cached")
        continue
    lane_source = SnapshotDataset(LANES[key].source.name, path="data")
    lane_rows[key] = rows
    lane_lookup[key] = QrelStore.from_dataset(lane_source).lookup(lane_source.name)
    print(f"{key:40s} loaded {len(rows):>6,} rows")


all_rows = [(key, row) for key, rows in lane_rows.items() for row in rows]
print(f"pooled oracle rows: {len(all_rows):,} across {len(lane_rows)} of {len(LANES)} lanes")
if skipped:
    print(f"not yet cached ({len(skipped)}): {', '.join(skipped)}")

In [ ]:
from tqdm.auto import tqdm


def relabel(make_objective, desc: str) -> pd.Series:
    # derive_route, not argmax: ties resolve to the cheapest route (SPEC d41),
    # matching exactly how labels.parquet's `route` column is derived — a plain
    # max() here would silently default every tie to dense_only (StrategyName's
    # enum order), corrupting the flip-rate comparison against `shipped`.
    picks = []
    for key, row in tqdm(all_rows, desc=desc):
        objective = make_objective(LANES[key].min_relevance)
        gold = lane_lookup[key].get(row.query_id, {})
        scored = {
            route: objective.assess(
                {d: 1.0 / (i + 1) for i, d in enumerate(ids)}, gold
            )[0]
            for route, ids in row.route_rankings.items()
        }
        picks.append(str(derive_route(scored)))
    return pd.Series(picks)


shipped = pd.Series([str(row.strategy_name) for _, row in all_rows])
variants = {
    "1·HR@1": lambda mr: RouterObjective(hit_weight=1, ndcg_weight=0, min_relevance=mr),
    "0.7·HR@1 + 0.3·NDCG@10  (shipped)": lambda mr: RouterObjective(min_relevance=mr),
    "0.5·HR@1 + 0.5·NDCG@10": lambda mr: RouterObjective(hit_weight=0.5, ndcg_weight=0.5, min_relevance=mr),
    "bare NDCG@10 (no top-1 term)": lambda mr: NDCGObjective(min_relevance=mr),
    "shipped, stricter min_relevance+1": lambda mr: RouterObjective(min_relevance=mr + 1),
}

rows_out = []
for name, make_objective in variants.items():
    picks = relabel(make_objective, desc=name)
    result = {
        "objective": name,
        "flips": int((picks != shipped).sum()),
        "flip rate": f"{(picks != shipped).mean() * 100:.1f}%",
        **{f"picks {r}": int((picks == r).sum())
           for r in ("dense_only", "pure_rrf", "sparse_only")},
    }
    print(f"  -> {result['flips']:,} flips ({result['flip rate']})")
    rows_out.append(result)
show(pd.DataFrame(rows_out))

## 10 — Where the golden set stands

Re-read coverage now that one dataset is labelled, and project the trainable
yield. The projection assumes other datasets behave like this one, which they
will not — nfcorpus is a single 3,633-document medical corpus, unusually
richly judged. Treat it as an order of magnitude, not a forecast.

In [ ]:
coverage = labels.coverage()
show(coverage[coverage.labelled > 0])

done = labels.load()
trainable = (done["shape"] == "routes_differ").mean()
print(f"labelled so far:        {len(done):,} of {len(selection):,} rows")
print(f"trainable share:        {trainable * 100:.1f}%")
print(f"projected yield at 50K: ~{int(trainable * len(selection)):,} rows "
      f"(extrapolated from one dataset — see caveat above)")
print()
print("next unblock, largest first:")
show(coverage[coverage.labelled == 0].head(5)[["dataset", "selected", "unlabelled"]])

## 11 — msmarco-passage-dev: the composition's largest lane (SPEC d38)

15,678 composition rows — 31% of the 50K. The local dev qrels cover 7,697 of
them (49.1%); the rest stay `unlabelled` in coverage — a gap to report, never
a licence to substitute other queries. The gap is MS MARCO's own: the full
dev set ships 101,093 queries but judgments were released for only 55,578
(55%), and the fill drew judgment-blind — the composition's `checkable=True`
was assigned per-dataset, not per-query. Median **one** judged passage per
query against nfcorpus's 16, so this lane sits at the opposite end of the
judgment-density axis: the regime where two different top-10 lists cannot
both be right.

The corpus recipe (d38c): every judged-relevant passage for the selected
queries is force-included, then padded with uniform-random passages from the
full 8.8M collection to **100,000** total, fixed seed. Uniform sampling
preserves the collection's vocabulary/IDF profile — the d37(g) fix: trec-dl's
judged-docs-only corpus was near-all answers, which inflates dense and
starves sparse.

Materialization is one-time and local (the ir_datasets collection is already
on disk; first `docs_store` access builds its index). Re-runs read the
snapshot back via `SnapshotDataset`.

In [ ]:
from pathlib import Path

from hybrid_search_rrf_dataset.retrieval import MSMarcoDev

MS_DATASET = "msmarco-passage-dev"      # composition key == source name here
MS_COLLECTION = "msmarco_routes"

if not (Path("data") / MS_DATASET / "corpus.parquet").exists():
    ms = MSMarcoDev(
        query_ids=labels.rows_for(MS_DATASET)["query_id"],
        corpus_size=100_000,
        seed=0,                          # d38(c): fixed seed, recipe is a parameter
    )
    ms.materialize()
    ms.save("data")

ms_source = SnapshotDataset(MS_DATASET, path="data")
ms_corpus, ms_queries, ms_qrels = ms_source.corpus(), ms_source.queries(), ms_source.qrels()
print(f"corpus  {len(ms_corpus):,} passages "
      f"({ms_qrels.doc_id.nunique():,} judged-relevant, rest uniform distractors)")
print(f"queries {len(ms_queries):,} of {len(labels.rows_for(MS_DATASET)):,} composition rows "
      f"({len(ms_queries) / len(labels.rows_for(MS_DATASET)) * 100:.1f}% have dev qrels)")
print(f"qrels   {len(ms_qrels):,} judgments, grades {sorted(ms_qrels.relevance.unique())}")

## 12 — Index the 100K corpus

Same two named vector slots as every route collection. The one-time cost is
the dense pass over 100K passages (order of an hour on this machine); the
embedding cache makes re-runs cheap, and the upload skips when the collection
is already full.

In [ ]:
ms_indexer = CorpusIndexer(
    client, MS_COLLECTION,
    embeddings=[dense_cfg, sparse_cfg],
    cache=EmbeddingCache("./.embedding_cache"),
)
ms_indexer.ensure_collection()

if client.count(MS_COLLECTION, exact=True).count >= len(ms_corpus):
    print(f"{MS_COLLECTION}: already indexed — skipping upload")
else:
    ms_indexer.upload(
        [CorpusDocument(**r) for r in ms_corpus.to_dict("records")], batch_size=64
    )
print(f"{MS_COLLECTION}: {client.count(MS_COLLECTION, exact=True).count:,} points")

## 13 — Label the lane

Same argmax rule as the nfcorpus anchor (d38e), so the two datasets differ by
exactly one variable — the index they are scored on. `RouteLabels.label`
narrows the 15,678 selection rows to the snapshot's queries via `QuerySubset`
and merges only this dataset's rows into `labels.parquet`. `min_relevance=1`
holds: the dev qrels are binary.

~7,700 queries × 3 routes; at nfcorpus throughput this is on the order of ten
minutes against local Qdrant.

In [ ]:
ms_args = (client, MS_COLLECTION, dense_cfg, sparse_cfg)
ms_dense, ms_hybrid, ms_sparse = (
    DenseOnlyStrategy(*ms_args), PureRRFStrategy(*ms_args), SparseOnlyStrategy(*ms_args)
)

ms_labelled = labels.label(ms_source, ms_dense, ms_hybrid, ms_sparse, dataset=MS_DATASET)
print(f"labelled {len(ms_labelled):,} rows -> {labels.labels_path}")
show(ms_labelled.head(8).round(3))

## 14 — Outcome shapes on a realistic index

Same three shapes as §7, now on a corpus that is 92% distractors. With a
median of **one** judged passage per query, `all_tied` requires all three
routes to place that same passage at the same effective rank, and
`routes_differ` means at least one route actually found it while another
did not — a much harder tie regime than nfcorpus's.

In [ ]:
ms_shapes = (ms_labelled.groupby(["cell", "shape"]).size()
             .unstack(fill_value=0))
ms_shapes["total"] = ms_shapes.sum(axis=1)
show(ms_shapes.reset_index())

overall = ms_labelled["shape"].value_counts().rename_axis("shape").reset_index(name="rows")
overall["share"] = (overall.rows / len(ms_labelled) * 100).round(1).astype(str) + "%"
show(overall)

signal = ms_labelled[ms_labelled["shape"] == "routes_differ"]
dist = signal.route.value_counts().rename_axis("route").reset_index(name="rows")
dist["share"] = (dist.rows / len(signal) * 100).round(1).astype(str) + "%"
print(f"route distribution over the {len(signal):,} trainable rows:")
show(dist)
print(f"majority-class baseline: {dist.rows.iloc[0] / len(signal) * 100:.0f}% "
      f"(always predict {dist.route.iloc[0]})")

## 15 — Where the golden set stands, two datasets in

The open label-form question (argmax one-hot vs the per-route score vector,
TODOS) turns on the **margin** — winner's score minus runner-up's. A fat
margin is a fact about retrieval; a thin one is a coin toss the objective
happened to break, and it flips when the encoder or corpus changes.
`decisive` counts rows with margin ≥ 0.06 *and* a rank-1 hit — roomier than
any NDCG-tail wiggle, and excluding "least bad" wins where every route missed.

nfcorpus margins were thin because 86% of its corpus is judged relevant to
*something* — two disjoint top-10s can both be right. msmarco's
median-1-relevant regime is the counter-test: either a route surfaced the one
judged passage or it scored zero, so ties require actually retrieving the
same passage at the same rank.

In [ ]:
SCORES = ["score_dense_only", "score_pure_rrf", "score_sparse_only"]

done = labels.load()
done["margin"] = done[SCORES].max(axis=1) - done[SCORES].apply(
    lambda r: sorted(r)[-2], axis=1
)
done["hit"] = done[SCORES].max(axis=1) >= 0.7

rows = []
for ds, group in done.groupby("dataset"):
    differ = group[group["shape"] == "routes_differ"]
    decisive = (differ.margin >= 0.06) & differ.hit
    rows.append({
        "dataset": ds,
        "labelled": len(group),
        "routes_differ": len(differ),
        "median margin": round(differ.margin.median(), 3),
        "p90 margin": round(differ.margin.quantile(0.9), 3),
        "decisive": int(decisive.sum()),
        "decisive share": f"{decisive.mean() * 100:.0f}%",
    })
show(pd.DataFrame(rows))

coverage = labels.coverage()
show(coverage[coverage.labelled > 0])
print(f"labelled {coverage.labelled.sum():,} of {coverage.selected.sum():,} "
      f"({coverage.labelled.sum() / coverage.selected.sum() * 100:.1f}%)")

## 16 — Readiness: every lane materialized, only indexing remains

The two-pass acquisition (qrels, then corpus) is complete — all 42 lanes were
built by `materialize_corpora.py`, so `cell_selection` needs no fetch. What's
left is indexing each corpus into Qdrant, then labelling.

This cell is the progress bar and defines the helpers the sweep reuses:
`collection_of` maps a composition key to its route collection (with the one
pre-convention name, `msmarco_routes`, overridden so the 100K index is reused
not rebuilt), and `datasets` is every selected dataset ordered **cheapest
corpus first** — the sweep's order.

In [ ]:
from pyarrow.parquet import ParquetFile

from hybrid_search_rrf_dataset.lanes import LANES

# the two anchor collections predate the {source.name}_routes convention
# (§4, §12); override so the sweep reuses their indexes, not re-embeds under
# new names.
COLLECTION_OVERRIDE = {
    "beir-nfcorpus": "nfcorpus_routes",
    "msmarco-passage-dev": "msmarco_routes",
}


def source_name(key: str) -> str:
    return LANES[key].source.name if key in LANES else key


def collection_of(key: str) -> str:
    return COLLECTION_OVERRIDE.get(key, f"{source_name(key)}_routes")


def corpus_rows(key: str) -> int | None:
    path = Path("data") / source_name(key) / "corpus.parquet"
    return ParquetFile(path).metadata.num_rows if path.exists() else None


# every selected dataset, cheapest corpus first — the sweep's order
datasets = sorted(
    selection["dataset"].astype(str).unique(),
    key=lambda k: corpus_rows(k) if corpus_rows(k) is not None else float("inf"),
)
live = {c.name for c in client.get_collections().collections}
done = set(labels.load()["dataset"].unique()) if labels.labels_path.exists() else set()

ready = pd.DataFrame(
    [
        {
            "dataset": k,
            "corpus_rows": corpus_rows(k),
            "indexed": collection_of(k) in live,
            "labelled": k in done,
        }
        for k in datasets
    ]
)
show(ready)
print(
    f"datasets {len(ready)} | corpus on disk {ready.corpus_rows.notna().sum()} "
    f"| indexed {int(ready.indexed.sum())} | labelled {int(ready.labelled.sum())}"
)

In [ ]:
coverage = labels.coverage()
show(coverage)
print(f"selected {coverage.selected.sum():,} | "
      f"labelled {coverage.labelled.sum():,} | "
      f"qrels_ready {coverage.qrels_ready.sum():,} | "
      f"unlabelled {coverage.unlabelled.sum():,}")

## 17 — Sweep: index and label every remaining lane, cheapest first

One loop over all 42 datasets in `datasets` order (§16, size-ascending), so the
cheap lanes label first and a long dense pass never blocks quick wins. Per lane:
read the on-disk corpus → index into its route collection if not already full →
label with the argmax objective. Every step idempotent: a full collection skips
the upload, and a dataset already in `labels.parquet` is skipped whole.

- **`min_relevance` is per-lane** from `LANES` — antique binarizes at grade 3
  (its level 2 is "does not answer"), trec-dl at 2, the rest at 1. A fresh
  `RouteLabels` per lane carries the right threshold into the label and its
  stored `min_relevance`.
- **Corpus sizing already happened** in `materialize_corpora.py` (the d39 20/80
  `CorpusRecipe`, per-lane exceptions pinned in `LANES`), so the sweep only reads
  snapshots — no `materialize()` here.
- **BRIGHT `excluded_ids`** ride on each snapshot's `excluded.parquet` and are
  applied per query at scoring time inside `RouteLabels.label`.
- **One bad lane never aborts the sweep** — a lane whose selected queries have
  no judgments is caught and reported, and the loop continues.

> Starting a clean golden set? `labels.parquet` may still hold rows from the old
> `selection.parquet` query set. Delete it first
> (`rm data/route_labels/labels.parquet`) so labels aren't a mix of two
> compositions; the sweep then rebuilds every lane.

In [ ]:
already = set(labels.load()["dataset"].unique()) if labels.labels_path.exists() else set()

for key in datasets:                         # size-ascending, from §16
    if key in already:
        print(f"{key:38s} labelled — skip")
        continue

    name = source_name(key)
    src = SnapshotDataset(name, path="data")
    corpus = src.corpus()
    collection = collection_of(key)

    lane_indexer = CorpusIndexer(
        client, collection,
        embeddings=[dense_cfg, sparse_cfg],
        cache=EmbeddingCache("./.embedding_cache"),
    )
    lane_indexer.ensure_collection()
    if client.count(collection, exact=True).count < len(corpus):
        lane_indexer.upload(
            [CorpusDocument(**r) for r in corpus.to_dict("records")], batch_size=64
        )

    # per-lane threshold; a fresh RouteLabels shares the same labels.parquet
    min_rel = LANES[key].min_relevance if key in LANES else 1
    lane_labels = RouteLabels(selection, objective=RouterObjective(min_relevance=min_rel))

    strat_args = (client, collection, dense_cfg, sparse_cfg)
    try:
        out = lane_labels.label(
            src,
            DenseOnlyStrategy(*strat_args),
            PureRRFStrategy(*strat_args),
            SparseOnlyStrategy(*strat_args),
            dataset=key,
        )
    except ValueError as error:              # no judged queries in this lane's selection
        print(f"{key:38s} SKIPPED: {error}")
        continue

    shp = out["shape"].value_counts()
    print(f"{key:38s} corpus {len(corpus):>7,}  min_rel {min_rel}  "
          f"labelled {len(out):>6,}  differ {shp.get('routes_differ', 0):,} "
          f"| tied {shp.get('all_tied', 0):,} | zero {shp.get('all_zero', 0):,}")

In [ ]:
coverage = labels.coverage()
show(coverage[coverage.labelled > 0])
print(f"labelled {coverage.labelled.sum():,} of {coverage.selected.sum():,} | "
      f"qrels_ready {coverage.qrels_ready.sum():,}")

done = labels.load()
done["margin"] = done[SCORES].max(axis=1) - done[SCORES].apply(
    lambda r: sorted(r)[-2], axis=1
)
done["hit"] = done[SCORES].max(axis=1) >= 0.7

rows = []
for ds, group in done.groupby("dataset"):
    differ = group[group["shape"] == "routes_differ"]
    decisive = (differ.margin >= 0.06) & differ.hit
    dist = differ.route.value_counts()
    rows.append({
        "dataset": ds,
        "labelled": len(group),
        "routes_differ": len(differ),
        "decisive": int(decisive.sum()),
        "decisive share": f"{decisive.mean() * 100:.0f}%",
        "median margin": round(differ.margin.median(), 3) if len(differ) else None,
        "top route": dist.index[0] if len(dist) else None,
        "top share": f"{dist.iloc[0] / len(differ) * 100:.0f}%" if len(differ) else None,
    })
show(pd.DataFrame(rows).sort_values("labelled", ascending=False))

## 18 — How much is routing worth? The headroom readout

Three levels, same labelled rows:

1. **One global constant** — always answer with the single best method, everywhere.
2. **Best constant per collection** — someone tells you, per dataset, which one method
   works best there, and you follow it blindly.
3. **Per-query oracle** — a fortune-teller picks the best method for every individual
   query. This is the ceiling.

The 1→2 gap is what *knowing your collection* is worth; 2→3 is what *judging each query*
adds. Together they are the business case for a router.

**Read it with the fine print, always:**

- The oracle is a **ceiling, not an achievement**. Published attempts at this kind of
  per-query selection achieved ~4% at best — a router capturing even half of our ceiling
  would be exceptional.
- The pooled number depends on the **dataset mix**, which was never designed for routing:
  these queries were composed for feature diversity. **This is not an optimal routing
  dataset**, and its pooled headroom must not be quoted without this caveat.
- Judgment holes are unmeasured and bias scores against dense; every number is pinned to
  the current embedding stack (bge-small).
- **One-sided decisiveness is not headroom.** `limit` is 43% decisive with ~0 headroom:
  one method wins every decisive row there, so the constant already captures it — visible
  in the winner table below.

Decisive = the winner put a relevant document at rank 1 and the runner-up did not; the
margin threshold is derived from the objective's weights, never hand-typed.

Ratified decision and full caveat list: SPEC.md decision 44.

In [ ]:
show(labels.headroom_decomposition().round(3))

headroom = labels.headroom()
show(headroom)

winners = labels.decisive_winners()
show(winners)

pooled = headroom[headroom.dataset == "POOLED"].iloc[0]
mix = headroom[headroom.dataset != "POOLED"].nlargest(1, "labelled").iloc[0]
print(f"ceiling over one global constant: +{pooled.headroom_pct}% "
      f"({pooled.constant:.3f} -> {pooled.oracle:.3f})")
print(f"lane-mix warning: {mix.dataset} alone is "
      f"{mix.labelled / pooled.labelled * 100:.0f}% of labelled rows "
      f"and its own ceiling is only +{mix.headroom_pct}%")

## 19 — Route signal per cell (the prior, tested)

The point of the cell composition: does each archetype's a-priori `predicts`
route match what retrieval actually rewards? This joins the golden labels back
to `cell_selection` on `query_id` — the **exact** per-cell view, so a query
filling several cells counts under each — and sets the measured top route
(over the trainable `routes_differ` rows) against the cell's declared prior.

`prior_holds` is the falsification check: where measured ≠ predicted, either the
cell's prior is wrong or the lanes labelled so far are skewed. **Read it after
the sweep finishes** — mid-sweep the cheap, dense-friendly lanes dominate, so
almost everything reads `dense_only` until the technical/entity lanes land.

In [ ]:
from composition.cells import CELLS

# exact per-cell view: re-join full cell membership so a query that fills
# several cells counts under each of them (the §7 markdown's promise)
cell_map = (
    selection[["dataset", "query_id", "cell"]]
    .astype({"query_id": str})
    .drop_duplicates()
)
done = labels.load()
done["query_id"] = done["query_id"].astype(str)
per_cell = done.drop(
    columns=[c for c in ("cell", "stage", "route_selected") if c in done]
).merge(cell_map, on=["dataset", "query_id"], how="left")

predicts = {cell.name: set(cell.predicts) for cell in CELLS}
rows = []
for cell, group in per_cell.groupby("cell"):
    differ = group[group["shape"] == "routes_differ"]
    dist = differ["route"].value_counts()
    top = dist.index[0] if len(dist) else None
    rows.append({
        "cell": cell or "(control)",
        "labelled": len(group),
        "trainable %": f"{len(differ) / len(group) * 100:.0f}%" if len(group) else "0%",
        "dense": int((differ["route"] == "dense_only").sum()),
        "rrf": int((differ["route"] == "pure_rrf").sum()),
        "sparse": int((differ["route"] == "sparse_only").sum()),
        "measured": top,
        "predicted": ",".join(sorted(predicts.get(cell, ()))) or "-",
        "prior_holds": (top in predicts.get(cell, set())) if top else None,
    })
per_cell_tbl = pd.DataFrame(rows).sort_values("labelled", ascending=False)
show(per_cell_tbl)

held = per_cell_tbl["prior_holds"].dropna()
print(f"predicted route is the measured top route in {int(held.sum())}/{len(held)} cells "
      f"— provisional: dense-friendly cheap lanes dominate until the sweep finishes")